# 01: Load and clean the match data

Combine the Premier League season files into one clean, chronologically sorted dataset and save it to `data/processed/matches.csv`.

**Source:** [football-data.co.uk](https://www.football-data.co.uk/englandm.php), one CSV per season (`E1718.csv`, `E1819.csv`, ...). The raw files are not stored in the repository.

**Output:** `data/processed/matches.csv`, with 3,520 matches and 22 columns, from the opening day of 2017/18 (11 Aug 2017) to 8 Nov 2026. That is nine full seasons plus the first 100 matches of 2026/27.

**Contents**
1. Load every season into one table
2. Keep the columns needed for modeling
3. Parse and check the dates
4. Sort chronologically
5. Quality checks: duplicates, missing values, team names
6. Save the cleaned data

## 1. Load every season into one table

Read each Premier League CSV in `data/raw` (files named `E*.csv`), drop blank rows, tag every row with its season code (`1718` means 2017/18), and stack all seasons into one `matches` table.

Three details matter here:
- `encoding="latin-1"` avoids a decoding error, because some older football-data files are not UTF-8.
- `dropna(subset=["HomeTeam"])` removes empty rows that some files have at the end.
- The folder is read with a relative path (`../data/raw`), so the notebook works on any machine as long as it is run from the `notebooks/` folder.

In [1]:
from pathlib import Path
import pandas as pd

raw_dir = Path("../data/raw")
frames = []

for file in sorted(raw_dir.glob("E*.csv")):
    season = file.stem[1:]                    # "E1718" -> "1718"
    df = pd.read_csv(file, encoding="latin-1")
    df = df.dropna(subset=["HomeTeam"])       # remove blank rows
    df["Season"] = season
    frames.append(df)

matches = pd.concat(frames, ignore_index=True)

### Table size

Show how many matches (rows) and columns are in the combined table.

In [2]:
matches.shape

(3520, 196)

**Result:** 3,520 rows and 196 columns. The column count is high because every football-data file carries dozens of bookmaker odds columns, and stacking the seasons keeps the union of all of them. Most are not needed.

### Matches per season

Count the matches in each season code. A full Premier League season has 380 matches (20 teams x 19 opponents), so every completed season should show exactly 380.

In [3]:
matches['Season'].value_counts()

Season
1718    380
1819    380
1920    380
2021    380
2122    380
2223    380
2324    380
2425    380
2526    380
2627    100
Name: count, dtype: int64

**Result:** the nine completed seasons (2017/18 to 2025/26) each have exactly 380 matches, so no season is missing or cut short. 2026/27 is in progress and has 100 matches so far.

## 2. Keep the columns needed for modeling

Keep 22 columns and drop the rest:
- **Keys:** season, date, home team, away team
- **Result:** full-time home goals (`FTHG`), away goals (`FTAG`) and result (`FTR`: H, D or A)
- **Match statistics:** shots (`HS`, `AS`), shots on target (`HST`, `AST`), fouls (`HF`, `AF`), corners (`HC`, `AC`), yellow cards (`HY`, `AY`) and red cards (`HR`, `AR`)
- **Bookmaker odds:** Bet365 home, draw and away odds (`B365H`, `B365D`, `B365A`)

Shots, fouls, corners and cards are only known after the match, so they can never be used as features for that same match. The odds are kept as a benchmark to compare the model against, not as a model input.

In [4]:
core_cols = [
    "Season", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HS", "AS", "HST", "AST",
    "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR",
    "B365H", "B365D", "B365A",
]
matches = matches[core_cols]

### Preview the table

Display the table after the extra columns were removed. At this point `Date` is still text in day/month/year form (for example `11/08/2017`), which is why the next step parses it.

In [5]:
matches

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,...,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A
0,1718,11/08/2017,Arsenal,Leicester,4,3,H,27,6,10,...,12,9,4,0,1,0,0,1.53,4.50,6.50
1,1718,12/08/2017,Brighton,Man City,0,2,A,6,14,2,...,9,3,10,0,2,0,0,11.00,5.50,1.33
2,1718,12/08/2017,Chelsea,Burnley,2,3,A,19,10,6,...,11,8,5,3,3,2,0,1.25,6.50,15.00
3,1718,12/08/2017,Crystal Palace,Huddersfield,0,3,A,14,8,4,...,19,12,9,1,3,0,0,1.83,3.60,5.00
4,1718,12/08/2017,Everton,Stoke,1,0,H,9,9,4,...,10,6,7,1,1,0,0,1.70,3.80,5.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3515,2627,08/11/2026,Brighton,Brentford,2,0,H,20,5,6,...,7,7,3,4,2,0,1,1.90,3.80,3.70
3516,2627,08/11/2026,Crystal Palace,Liverpool,1,3,A,19,20,7,...,18,4,7,2,4,0,0,5.50,4.33,1.57
3517,2627,08/11/2026,Ipswich,Bournemouth,1,2,A,10,15,2,...,20,2,7,2,3,0,0,3.80,3.75,1.91
3518,2627,08/11/2026,Sunderland,Chelsea,1,0,H,9,10,2,...,8,2,2,5,4,0,0,3.90,3.60,1.91


### Look at the in-progress season

Show only the rows whose season code is `2627`. This is the current season, with 100 matches played up to 8 Nov 2026. It is incomplete, so later notebooks keep it out of the training, validation and test sets.

In [6]:
matches[matches['Season']== '2627']

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,...,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A
3420,2627,21/08/2026,Arsenal,Coventry,3,0,H,20,4,6,...,13,8,2,1,1,0,0,1.20,7.00,13.00
3421,2627,22/08/2026,Hull,Man United,2,0,H,8,21,4,...,9,1,6,2,1,0,0,8.50,5.00,1.36
3422,2627,22/08/2026,Everton,Crystal Palace,2,0,H,16,11,5,...,15,4,0,0,1,0,0,2.15,3.50,3.30
3423,2627,22/08/2026,Ipswich,Sunderland,2,1,H,10,11,3,...,14,3,5,0,3,0,0,2.70,3.30,2.63
3424,2627,22/08/2026,Nott'm Forest,Leeds,0,1,A,12,11,2,...,14,3,2,2,1,0,0,2.30,3.40,3.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3515,2627,08/11/2026,Brighton,Brentford,2,0,H,20,5,6,...,7,7,3,4,2,0,1,1.90,3.80,3.70
3516,2627,08/11/2026,Crystal Palace,Liverpool,1,3,A,19,20,7,...,18,4,7,2,4,0,0,5.50,4.33,1.57
3517,2627,08/11/2026,Ipswich,Bournemouth,1,2,A,10,15,2,...,20,2,7,2,3,0,0,3.80,3.75,1.91
3518,2627,08/11/2026,Sunderland,Chelsea,1,0,H,9,10,2,...,8,2,2,5,4,0,0,3.90,3.60,1.91


## 3. Parse and check the dates

Turn the `Date` text into real datetimes. `dayfirst=True` makes `03/02/2018` mean 3 February, not 2 March, and `format="mixed"` accepts both `dd/mm/yy` and `dd/mm/yyyy`, which both appear across the files.

Getting the day and month the wrong way round does not raise an error, so the result is checked in three ways below: the data type, the number of dates that failed to parse, and the date range of every season.

In [7]:
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True, format="mixed")

### Check the dates parsed

Confirm that `Date` is a datetime column and that no date failed to parse (the missing count should be 0).

In [8]:
matches["Date"].dtype          # should be datetime64[ns]
matches["Date"].isna().sum()   # should be 0

np.int64(0)

**Result:** 0 dates failed to parse.

### Date range of each season

For every season, show the earliest and latest match date. If day and month had been swapped, seasons would not run from August to May.

In [9]:
matches.groupby("Season")["Date"].agg(["min", "max"])

,min,max
Season,,
1718,2017-08-11,2018-05-13
1819,2018-08-10,2019-05-12
1920,2019-08-09,2020-07-26
2021,2020-09-12,2021-05-23
2122,2021-08-13,2022-05-22
2223,2022-08-05,2023-05-28
2324,2023-08-11,2024-05-19
2425,2024-08-16,2025-05-25
2526,2025-08-15,2026-05-24


**Result:** every season starts in August (or September in 2020) and ends in May, with two exceptions caused by the COVID-19 pandemic: 2019/20 ended on 26 July 2020, and 2020/21 started late, on 12 September 2020. 2026/27 runs from 21 Aug 2026 to 8 Nov 2026 so far. No season has odd months, so day and month were read the right way round.

### Date range of the whole dataset

Show the earliest and latest match date across all seasons.

In [10]:
matches["Date"].min(), matches["Date"].max()

(Timestamp('2017-08-11 00:00:00'), Timestamp('2026-11-08 00:00:00'))

**Result:** from 11 Aug 2017 to 8 Nov 2026.

## 4. Sort chronologically

Order the table from the oldest to the newest match and reset the row numbers. Chronological order matters for the whole project: rolling-form features must only look at earlier matches, and the train/validation/test split is made by time.

In [11]:
matches = matches.sort_values("Date").reset_index(drop=True)

## 5. Quality checks

### Duplicate matches

Count rows that are identical in every column, and rows that repeat the same date with the same home and away team.

In [12]:
matches.duplicated().sum()                                   # fully identical rows
matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()   # same match twice

np.int64(0)

**Result:** 0. The display shows only the last line of the cell (the same-match-twice check), but a fully identical row would also repeat the date and both teams, so there are no duplicates of either kind.

### Missing values

Show how many missing values each column has.

In [13]:
matches.isna().sum()

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HS          0
AS          0
HST         0
AST         0
HF          0
AF          0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
B365H       0
B365D       0
B365A       0
dtype: int64

**Result:** 0 in every one of the 22 columns, including shots, cards and the Bet365 odds, so no rows need to be dropped or filled.

### Team names

List every home team to look for the same club spelled two different ways, and count the distinct home teams per season (it should be 20).

In [14]:
matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()   # expect 0
matches.groupby("Season")["HomeTeam"].nunique()                     # expect 20 per full season
sorted(set(matches["HomeTeam"]))                                    # read the list for spelling variants

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Burnley',
 'Cardiff',
 'Chelsea',
 'Coventry',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Huddersfield',
 'Hull',
 'Ipswich',
 'Leeds',
 'Leicester',
 'Liverpool',
 'Luton',
 'Man City',
 'Man United',
 'Newcastle',
 'Norwich',
 "Nott'm Forest",
 'Sheffield United',
 'Southampton',
 'Stoke',
 'Sunderland',
 'Swansea',
 'Tottenham',
 'Watford',
 'West Brom',
 'West Ham',
 'Wolves']

**Result:** 34 distinct teams, each spelled one way throughout, using football-data's short names (for example `Man City`, `Man United`, `Nott'm Forest`). These exact spellings are what later notebooks and the demo must use.

Jupyter only displays the last expression of a cell, so the duplicate count and the teams-per-season table above the team list did not show in the saved run. Re-checking on the saved file: every season, including the in-progress 2026/27, has exactly 20 distinct home teams. The result column also agrees with the goals in all 3,520 matches (`H` when the home goals exceed the away goals, and so on).

## 6. Save the cleaned data

Sort by date and then by home team, so the order is identical every time the notebook runs (many matches share a date), and write the table to `data/processed/matches.csv`.

CSV files do not remember data types, so later notebooks read the file with `parse_dates=["Date"]` and `dtype={"Season": str}`.

In [15]:
matches = matches.sort_values(["Date", "HomeTeam"]).reset_index(drop=True)

from pathlib import Path
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
matches.to_csv(out_dir / "matches.csv", index=False)

## Summary

**What this notebook did**
- Combined ten season files into one table of 3,520 matches
- Kept 22 columns: keys, result, match statistics and Bet365 odds
- Parsed the dates (day first) and checked each season's date range
- Sorted the matches chronologically
- Saved the cleaned data to `data/processed/matches.csv`

**Checks that passed**
- Nine completed seasons with exactly 380 matches each, plus 100 matches of 2026/27
- All dates parsed, and every season runs from August to May (apart from the COVID-affected 2019/20 and 2020/21)
- No duplicate matches and no missing values in any column
- 34 distinct teams with consistent spellings, 20 in every season
- The result column agrees with the goals scored in every match

**Things to remember later**
- 2026/27 is incomplete (100 matches), so it is kept out of training, validation and testing
- Shots, fouls, corners and cards are only known after the match, so they cannot be same-match features
- The odds are a benchmark, not a model input

**Next:** `02_eda.ipynb` explores the data and sets the baseline.